# Compare with Sutran paper baselines

In [8]:
import os
import numpy as np
import pandas as pd
from baselines.SuffixTransformerNetwork.results_collector import get_suffix_baseline_results

# ── Config ────────────────────────────────────────────────────────────────────

METRIC_COL = "MAE RRT (min) ↓"

OWN_CONFIGS = {
    "GNN_Time_MLP": {
        "results_sub": "approach_time/results",
        "csv_file":    "results_time_mlp.csv",
    },
    "GNN_Time_MLP_Edge": {
        "results_sub": "approach_time/results_edge",
        "csv_file":    "results_time_mlp_edge.csv",
    },
}

# ── Load own approach results (mean/std across runs) ──────────────────────────

def get_own_time_results():
    rows = []
    for model_name, cfg in OWN_CONFIGS.items():
        run_id, run_dfs = 1, []
        while True:
            path = os.path.join(cfg["results_sub"], f"run_{run_id}", cfg["csv_file"])
            if not os.path.isfile(path):
                break
            df = pd.read_csv(path)
            df[METRIC_COL] = df["mae_seconds"] / 60.0
            run_dfs.append(df)
            run_id += 1

        all_logs = {log for df in run_dfs for log in df["log"].unique()}
        for log_name in all_logs:
            row = {"Log": log_name, "Model": model_name, "Runs": len(run_dfs)}
            vals = [
                float(df.loc[df["log"] == log_name, METRIC_COL].iloc[0])
                for df in run_dfs
                if not df.loc[df["log"] == log_name].empty
            ]
            row[f"{METRIC_COL} mean"] = round(np.mean(vals),        2) if vals else float("nan")
            row[f"{METRIC_COL} std"]  = round(np.std(vals, ddof=1), 2) if len(vals) > 1 else float("nan")
            rows.append(row)

    return pd.DataFrame(rows).set_index(["Log", "Model"])

# ── Combine ───────────────────────────────────────────────────────────────────

logs = [
    d for d in os.listdir("baselines/SuffixTransformerNetwork/results_per_log")
    if os.path.isdir(os.path.join("baselines/SuffixTransformerNetwork/results_per_log", d))
]

df_baselines = get_suffix_baseline_results(logs)[[f"{METRIC_COL} mean", f"{METRIC_COL} std"]]
df_own       = get_own_time_results()

combined = pd.concat([df_baselines, df_own])
combined

# ── Per-log breakdown ─────────────────────────────────────────────────────────

for log_name, group in combined.groupby(level="Log"):
    print(f"\n=== Log: {log_name} ===")
    print(
        group.reset_index()[
            ["Model", "Runs", f"{METRIC_COL} mean", f"{METRIC_COL} std"]
        ].sort_values(f"{METRIC_COL} mean").to_string(index=False)
    )


=== Log: BPI Challenge 2017 ===
            Model   Runs  MAE RRT (min) ↓ mean  MAE RRT (min) ↓ std
     GNN_Time_MLP 1.0000            10438.1400                  NaN
GNN_Time_MLP_Edge 1.0000            10559.5800                  NaN
      SuTraN (DA)    NaN                   NaN                  NaN
     SuTraN (NDA)    NaN                   NaN                  NaN
   CRTP-LSTM (DA)    NaN                   NaN                  NaN
  CRTP-LSTM (NDA)    NaN                   NaN                  NaN
          ED-LSTM    NaN                   NaN                  NaN
         SEP-LSTM    NaN                   NaN                  NaN
             BEST    NaN                   NaN                  NaN

=== Log: BPIC15_1 ===
            Model   Runs  MAE RRT (min) ↓ mean  MAE RRT (min) ↓ std
  CRTP-LSTM (NDA)    NaN            37865.1300                  NaN
GNN_Time_MLP_Edge 1.0000            41237.5700                  NaN
     GNN_Time_MLP 1.0000            41453.4700              